# Upload of the release to Hugging Face

Uploads the release (arrays and tables, 39 files, about 104 GB) from Google Drive to the Hugging Face dataset
repository `REPO_ID`, with the dataset card. The MD5 of `checksums.md5` is checked first, so only the release it
identifies goes up. Each file is read from Drive once: it is copied to the local disk of the VM (`STAGING`) while its
MD5 is checked against `checksums.md5`, and the checked copy is uploaded. A file that does not match is not uploaded.
The copies of a batch are deleted once its commit is done. Nothing on Drive is changed or moved. The only writes
there are the new output folder below.

The repository is created private, and the notebook never makes it public. At the end of the run cell, the repository
is listed and compared with the release, `done.csv` and the card. When every file matches, it can be made public by
hand, on the settings page of the repository.

1. Use the Google account that holds the `ms-cofd-2dsl` folder (the same as for the raw archives). If it is only under
   "Shared with me", add a shortcut to My Drive first.
2. Put `README.md` into the output folder on Drive: the card, `docs/hf/release_README.md` of the pipeline.
3. Make a Hugging Face token with write access (huggingface.co, Settings, Access Tokens). The token cell asks for it.
   It stays in memory: it is not written to a file or printed. After a restart, run the token cell again.
4. Run all cells with `LIMIT = 1` as a test. The repository is created private, the card and `checksums.md5` are its
   first commit, and `DATACARD.md` follows. Check the repository page (only you can see it) and the verify cell: one
   file with no problem, all others missing. Then set `LIMIT = None` and run the run cell again.
5. When the Drive mount drops, the run cell mounts Drive again by itself and goes on. When the whole session
   disconnects, run all cells again: files in `done.csv` are skipped. Files that reached the repository without a row
   in `done.csv` are read again and recorded without a second upload.
6. At the end, the run cell reports that the repository holds the whole release, and the verify cell reports nothing
   missing, extra, mismatched or unchecked. The repository stays private until you make it public on its settings
   page.

A CPU runtime is enough. The largest file, `16x/train.h5`, is about 51 GB, so the staging disk needs about 53 GB free.
The settings cell prints the free space. When a batch does not fit, the run stops before copying it.

This notebook can run in a second Colab session next to the upload of the raw archives. It uses its own output and
staging folders and another repository.

In [ ]:
!pip -q install -U "huggingface_hub[hf_xet]>=0.30"
!mkdir -p tools && touch tools/__init__.py
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
RELEASE = '/content/drive/MyDrive/ms-cofd-2dsl/dataset/v1.0.0'    # holds checksums.md5 and the 38 files it lists
OUTPUT = '/content/drive/MyDrive/ms-cofd-2dsl/hf-upload-release'  # new folder; README.md goes here
CARD = OUTPUT + '/README.md'                                      # docs/hf/release_README.md of the pipeline
REPO_ID = 'asaydam/ms-cofd-2dsl'                                  # created private, made public by hand
RELEASE_MD5 = '82a46f931bb1d60b57dbc564305019fb'                  # MD5 of checksums.md5 of v1.0.0
STAGING = '/content/hf-stage-release'                             # local copies of one batch (not on Drive)
BATCH = 20                                                        # files per commit (and at most 10 GB)
WORKERS = 4                                                       # files read at the same time
LIMIT = 1                                                         # 1 for the test, None for all
print(f"{shutil.disk_usage('/content').free / 1e9:.0f} GB free on the local disk (about 53 GB needed)")

In [ ]:
from getpass import getpass
from huggingface_hub import HfApi
hf_token = getpass('Hugging Face token with write access: ')     # kept in memory only
print('logged in as', HfApi(token=hf_token).whoami()['name'])

In [ ]:
%%writefile tools/extract_line_samples.py
"""
Line samples of every run, read from the run archives (the sampled lines are kept only in the archives, and the wake
velocity within the obstacle's y extent is computed from them):

    python -m tools.extract_line_samples --archives <archives folder> --index archive_index.csv --output <folder>

Meant for Google Colab with Drive mounted (tools/colab/extract_line_samples.ipynb), so the 1.58 TB of archives are
read where they are. Each archive <run>/<case>.tar.zst is streamed (zstd and tar in streaming mode, nothing is
unpacked to disk). The five sets of the attribute run, <case>_attributes/postProcessing/lines/<time>/<set>_p_U.xy,
are kept, and the MD5 of the whole archive is checked against archive_index.csv. Only the live archives of the index
are read (the 22,500 runs of the dataset and the 18 base_lu runs of the dropped cases), not the superseded attempts.

Output:
    <output>/<run>/lines_<first case>_<n>.npz  for every case of the shard: <case>/time and, for every set,
                                               <case>/<set> of shape (points, 4): y (m), p, u_x, u_y (float32)
    <output>/done.csv                          run, case, time, md5_ok, bytes, shard: one row per finished archive
    <output>/errors.csv                        run, case, error

The points of a set are those OpenFOAM wrote: the wall-to-wall lines have all 2401, and the lines through the
centroid stop at the obstacle. A case in done.csv is skipped when the script runs again, so a disconnected Colab
session resumes where it stopped.

The Drive mount of Colab can drop during a long run ("Transport endpoint is not connected"). A shard is therefore
written to a local staging folder first, with its rows of done.csv next to it, and then copied to the output; when
the mount drops, the run stops with DriveLost and the staged shards are copied at the next start. A shard holds at
most shard_size cases and shard_bytes of archives, so that a drop loses little reading.

Needs numpy and zstandard.
"""

import argparse
import csv
import errno
import hashlib
import io
import os
import re
import shutil
import sys
import tarfile
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

SETS = ('wakeLine', 'centroidBelow', 'centroidAbove', 'inletSection', 'outletSection')
MEMBER = re.compile(r'(?:^|/)[^/]+_attributes/postProcessing/lines/([^/]+)/(\w+)_p_U\.xy$')
DONE_FIELDS = ['run', 'case', 'time', 'md5_ok', 'bytes', 'shard']
CHUNK = 1 << 22


class DriveLost(RuntimeError):
    """the mounted Drive stopped answering; remount it and run again"""


def mount_lost(error):
    return isinstance(error, OSError) and (error.errno == errno.ENOTCONN or 'Transport endpoint' in str(error))


class HashingReader:
    """file object that keeps the MD5 and the byte count of everything read through it"""

    def __init__(self, raw):
        self.raw, self.md5, self.bytes = raw, hashlib.md5(), 0

    def read(self, size=-1):
        data = self.raw.read(size)
        self.md5.update(data)
        self.bytes += len(data)
        return data

    def drain(self):
        while self.read(CHUNK):
            pass


def parse_set(text):
    """an OpenFOAM raw set (columns y, p, Ux, Uy, Uz) as float32 y, p, u_x, u_y"""
    data = np.loadtxt(io.BytesIO(text), dtype=np.float64, ndmin=2)
    return data[:, :4].astype(np.float32)


def read_archive(path):
    """
    the five line sets of one archive
    :return: (time, {set: array}, md5 of the archive, bytes)
    """
    import zstandard
    found = {}
    with open(path, 'rb') as raw:
        hashing = HashingReader(raw)
        reader = zstandard.ZstdDecompressor().stream_reader(hashing, read_size=CHUNK, read_across_frames=True)
        with tarfile.open(fileobj=reader, mode='r|') as tar:
            for member in tar:
                match = MEMBER.search(member.name)
                if match and member.isfile() and match.group(2) in SETS:
                    found.setdefault(match.group(1), {})[match.group(2)] = parse_set(tar.extractfile(member).read())
        hashing.drain()
    if len(found) != 1:
        raise ValueError(f"line sets of {len(found)} time folders: {sorted(found)}")
    (time_name, sets), = found.items()
    missing = [s for s in SETS if s not in sets]
    if missing:
        raise ValueError(f"sets missing: {missing}")
    return time_name, sets, hashing.md5.hexdigest(), hashing.bytes


def live_rows(index):
    """the live archives of archive_index.csv, i.e. the rows of <case>.tar.zst outside superseded/"""
    with open(index, newline='') as f:
        rows = [r for r in csv.DictReader(f)
                if r['file'] == f"{r['case']}.tar.zst" and '/superseded/' not in r['remote']]
    return sorted(rows, key=lambda r: (r['run'], r['case']))


def read_rows(path):
    if not path.exists():
        return []
    with open(path, newline='') as f:
        return list(csv.DictReader(f))


def append_rows(path, fields, rows):
    new = not path.exists()
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        if new:
            writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())


def shards_of(rows, shard_size, shard_bytes):
    """consecutive rows of one run, at most shard_size of them and shard_bytes of archives (at least one row)"""
    shards = []
    for row in rows:
        last = shards[-1] if shards else None
        if (last and last[0]['run'] == row['run'] and len(last) < shard_size
                and sum(int(r['bytes']) for r in last) + int(row['bytes']) <= shard_bytes):
            last.append(row)
        else:
            shards.append([row])
    return shards


def publish(staging, output):
    """copy the staged shards to the output and append their rows to done.csv; a shard leaves staging when copied"""
    for rows_path in sorted(staging.glob('*.csv')):
        npz = rows_path.with_suffix('.npz')
        rows = read_rows(rows_path)
        target = output / rows[0]['shard']
        try:
            target.parent.mkdir(exist_ok=True)
            shutil.copyfile(npz, target)
            append_rows(output / 'done.csv', DONE_FIELDS, rows)
        except OSError as error:
            if mount_lost(error):
                raise DriveLost(str(error)) from error
            raise
        npz.unlink()
        rows_path.unlink()


def run(archives, index, output, workers=6, shard_size=250, shard_bytes=20e9, limit=None, staging=None):
    """
    read every live archive not yet in done.csv
    :param limit: read at most this many archives (a test run)
    :param staging: local folder for finished shards before they are copied to the output
    :raise DriveLost: when the mounted Drive stops answering (the finished shards stay in staging)
    """
    archives, output = Path(archives), Path(output)
    staging = Path(staging or Path(tempfile.gettempdir()) / 'line-samples-staging')
    staging.mkdir(parents=True, exist_ok=True)
    try:
        output.mkdir(parents=True, exist_ok=True)
        publish(staging, output)
        done = {(r['run'], r['case']) for r in read_rows(output / 'done.csv')}
        todo = [r for r in live_rows(index) if (r['run'], r['case']) not in done]
    except OSError as error:
        if mount_lost(error):
            raise DriveLost(str(error)) from error
        raise
    if limit is not None:
        todo = todo[:limit]
    print(f"{len(done)} archives done, {len(todo)} to read", flush=True)

    def one(row):
        try:
            time_name, sets, md5, size = read_archive(archives / row['run'] / row['file'])
            return row, time_name, sets, md5 == row['md5'], size, None
        except Exception as error:                        # noqa: BLE001  (recorded, the other archives go on)
            return row, None, None, False, 0, error

    total_bytes, started = 0, time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for shard in shards_of(todo, shard_size, shard_bytes):
            arrays, done_rows, error_rows, lost = {}, [], [], None
            for row, time_name, sets, md5_ok, size, error in pool.map(one, shard):
                if error is not None:
                    if mount_lost(error):
                        lost = error                      # not an error of the archive: it is read again later
                    else:
                        error_rows.append({'run': row['run'], 'case': row['case'],
                                           'error': f"{type(error).__name__}: {error}"})
                    continue
                arrays[f"{row['case']}/time"] = np.array(time_name)
                for name, values in sets.items():
                    arrays[f"{row['case']}/{name}"] = values
                done_rows.append({'run': row['run'], 'case': row['case'], 'time': time_name, 'md5_ok': md5_ok,
                                  'bytes': size})
                total_bytes += size
            if done_rows:
                name = f"lines_{done_rows[0]['case']}_{len(done_rows)}"
                for r in done_rows:
                    r['shard'] = f"{shard[0]['run']}/{name}.npz"
                stem = staging / f"{shard[0]['run']}_{name}"
                np.savez_compressed(stem.with_suffix('.npz'), **arrays)
                with open(stem.with_suffix('.csv'), 'w', newline='') as f:    # written last: marks the shard complete
                    writer = csv.DictWriter(f, fieldnames=DONE_FIELDS)
                    writer.writeheader()
                    writer.writerows(done_rows)
            if lost is not None:
                raise DriveLost(str(lost)) from lost
            try:
                publish(staging, output)
                if error_rows:
                    append_rows(output / 'errors.csv', ['run', 'case', 'error'], error_rows)
            except OSError as error:
                if mount_lost(error):
                    raise DriveLost(str(error)) from error
                raise
            elapsed = time.time() - started
            bad = sum(not r['md5_ok'] for r in done_rows)
            print(f"{shard[0]['run']:>8s} {shard[0]['case']} ... {len(done_rows)} read, {len(error_rows)} errors, "
                  f"{bad} MD5 mismatches | {total_bytes / 1e9:.1f} GB in {elapsed / 3600:.2f} h "
                  f"({total_bytes / 1e6 / max(elapsed, 1):.0f} MB/s)", flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--archives', required=True, help="folder with <run>/<case>.tar.zst")
    parser.add_argument('--index', required=True, help="archive_index.csv of the production")
    parser.add_argument('--output', required=True)
    parser.add_argument('--workers', type=int, default=6)
    parser.add_argument('--shard-size', type=int, default=250)
    parser.add_argument('--shard-bytes', type=float, default=20e9)
    parser.add_argument('--limit', type=int, default=None)
    parser.add_argument('--staging', default=None)
    args = parser.parse_args()
    run(args.archives, args.index, args.output, args.workers, args.shard_size, args.shard_bytes, args.limit,
        args.staging)


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile tools/upload_raw_archives.py
"""
Upload of the run archives of the dataset to a public Hugging Face dataset repository, read where they are on
Google Drive:

    python -m tools.upload_raw_archives --archives <archives folder> --index archive_index.csv \
        --cases metadata.parquet --repo <user or org>/ms-cofd-2dsl-raw --output <folder> [--batch N] [--limit N]
    python -m tools.upload_raw_archives --index ... --cases ... --repo ... --output ... --verify-only

Meant for Google Colab with Drive mounted (tools/colab/upload_raw_archives.ipynb), like tools/extract_line_samples.py.
Only the 22,500 runs of the dataset are uploaded: the live archives of archive_index.csv (live_rows of
tools/extract_line_samples.py) of the cases in --cases, the case_name column of the release's metadata.parquet or
metadata.csv. The 18 dropped cases are not in the release, so their base_lu archives stay out; superseded attempts
are not live. Every case of the list must have a live archive of every run.

The repository:
    README.md               the dataset card (docs/hf/raw_README.md), with the licence
    index.csv               run, case, file, bytes, md5 of every archive of the dataset
    <run>/<case>.tar.zst    the archives

A new repository is created private. The card and index.csv are its first commit, before any archive, and only then
is it made public, so it is never public without its card and licence. A private repository is made public the same
way; one that is public already is left as it is. A card or index that differs from the one in the repository is
committed again at the next start, so an edited card is published by running the script again.

The archives follow in commits (HfApi.create_commit) of at most batch archives and batch_bytes of one run. Each
archive is read from Drive once: it is copied to a local staging folder, and its MD5 (checked against
archive_index.csv) and SHA-256 are computed in the same pass. The commit uploads the staged copy, so the uploaded
bytes are the checked ones; huggingface_hub's own SHA-256 of the copy is compared with ours. An archive whose MD5 does
not match is not uploaded and goes to errors.csv. The staged copies of a batch are removed when its commit is done
(or has failed), so at most one batch is on local disk. Nothing in the archives folder is changed
(HfApi.upload_large_folder is not used: it writes its cache into the folder it uploads).

Output (a new folder):
    <output>/done.csv       run, case, bytes, md5_ok, sha256, commit: one row per archive in the repository
                            (commit is empty for an archive that was found in the repository, see below)
    <output>/errors.csv     run, case, error
    <output>/verify.csv     path, run, case, problem, detail (--verify-only)

An archive in done.csv is skipped when the script runs again, so a disconnected Colab session resumes where it
stopped. The repository is listed once at the start. An archive it holds that done.csv does not (a commit that went
through while the session stopped) is read and recorded without uploading it again when its size and SHA-256 match.
When the Drive mount drops, the run stops with DriveLost: a partly copied archive is removed, and the batch is copied
again at the next start.

--verify-only lists the repository with the LFS metadata of its files (size and SHA-256, also for files stored with
xet) and compares every file with done.csv and the index. Missing, extra and mismatched files, and archives that are
in the repository but not in done.csv (unchecked), are printed and written to verify.csv. Nothing is uploaded.

Needs huggingface_hub >= 0.30 (hf_xet is used when it is installed), and pyarrow for a .parquet case list. The token
is that of run(token=...), or of huggingface_hub's login or HF_TOKEN; it is never written or printed.
"""

import argparse
import csv
import hashlib
import io
import os
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from tools.extract_line_samples import (CHUNK, DriveLost, HashingReader, append_rows, live_rows, mount_lost,
                                        read_rows, shards_of)

DONE_FIELDS = ['run', 'case', 'bytes', 'md5_ok', 'sha256', 'commit']
ERROR_FIELDS = ['run', 'case', 'error']
VERIFY_FIELDS = ['path', 'run', 'case', 'problem', 'detail']
INDEX_FIELDS = ['run', 'case', 'file', 'bytes', 'md5']        # index.csv of the repository: no remote paths
CARD, INDEX = 'README.md', 'index.csv'
REPO_TYPE = 'dataset'
CARD_FILE = Path(__file__).resolve().parent.parent / 'docs' / 'hf' / 'raw_README.md'


def case_names(path):
    """the cases of the dataset: the case_name column of the release's metadata.parquet or metadata.csv"""
    path = Path(path)
    if path.suffix == '.parquet':
        import pyarrow.parquet as pq
        return set(pq.read_table(path, columns=['case_name']).column('case_name').to_pylist())
    with open(path, newline='') as f:
        return {r['case_name'] for r in csv.DictReader(f)}


def select_rows(index, cases):
    """
    the archives to upload: the live archives of archive_index.csv of the cases of the dataset
    :param cases: the case names of the dataset (case_names)
    :return: (index rows sorted by run and case, one per run and case: the last one of an archive indexed twice;
              the (run, case) of the live archives of other cases, which are left out)
    :raise ValueError: when a case of the list has no live archive of one of the runs of the index
    """
    live = live_rows(index)
    runs = sorted({r['run'] for r in live})
    rows = {(r['run'], r['case']): r for r in live if r['case'] in cases}
    others = sorted({(r['run'], r['case']) for r in live if r['case'] not in cases})
    missing = [(run, case) for run in runs for case in sorted(cases) if (run, case) not in rows]
    if missing or not rows:
        raise ValueError(f"{len(missing)} runs of the {len(cases)} listed cases have no live archive in the index, "
                         f"e.g. {missing[:5]} (is the case list the release's metadata?)")
    return [rows[key] for key in sorted(rows)], others


def index_csv(rows):
    """index.csv of the repository: run, case, file, bytes, md5 of the archives"""
    text = io.StringIO()
    writer = csv.DictWriter(text, fieldnames=INDEX_FIELDS, extrasaction='ignore', lineterminator='\n')
    writer.writeheader()
    writer.writerows(rows)
    return text.getvalue().encode()


def repo_path(row):
    return f"{row['run']}/{row['file']}"


def staged_path(staging, row):
    return Path(staging) / row['run'] / row['file']


def stage_archive(source, target):
    """
    copies one archive to the staging folder and computes its MD5, SHA-256 and size in the same pass; the copy is
    written as <target>.part and renamed when complete, a partial copy is removed
    :return: (md5, sha256, bytes)
    """
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + '.part')
    sha256 = hashlib.sha256()
    try:
        with open(source, 'rb') as raw, open(partial, 'wb') as copy:
            hashing = HashingReader(raw)
            for data in iter(lambda: hashing.read(CHUNK), b''):
                sha256.update(data)
                copy.write(data)
        os.replace(partial, target)
    except BaseException:
        partial.unlink(missing_ok=True)
        raise
    return hashing.md5.hexdigest(), sha256.hexdigest(), hashing.bytes


def clear_staging(staging, rows):
    """removes the staged and partial copies of these rows, and the run folders they leave empty"""
    for row in rows:
        path = staged_path(staging, row)
        path.unlink(missing_ok=True)
        path.with_name(path.name + '.part').unlink(missing_ok=True)
    for run in {row['run'] for row in rows}:
        try:
            (Path(staging) / run).rmdir()
        except OSError:                                   # not empty, or not there
            pass


def check_staging(archives, staging):
    """the staging folder must be apart from the archives folder, since its copies are deleted"""
    archives, staging = Path(archives).resolve(), Path(staging).resolve()
    if staging == archives or archives in staging.parents or staging in archives.parents:
        raise ValueError(f"the staging folder {staging} must be outside the archives folder {archives}")


def repo_tree(api, repo):
    """{path: file} of every file of the repository, from one listing; LFS and xet files have lfs.size, lfs.sha256"""
    return {item.path: item for item in api.list_repo_tree(repo, repo_type=REPO_TYPE, recursive=True)
            if hasattr(item, 'size')}                    # folders have no size


def lfs_of(item):
    """(size, sha256) of a file of the listing; sha256 is None for a file not stored with LFS or xet"""
    if getattr(item, 'lfs', None) is None:
        return item.size, None
    return item.lfs.size, item.lfs.sha256


def same_content(item, data):
    """True when a file of the listing holds these bytes (its LFS SHA-256, or its git blob id)"""
    size, sha256 = lfs_of(item)
    if sha256 is not None:
        return size == len(data) and sha256 == hashlib.sha256(data).hexdigest()
    return item.blob_id == hashlib.sha1(b'blob %d\0' % len(data) + data).hexdigest()


def drive_answers(archives, error=None):
    """raises DriveLost when the error, or a listing of the archives folder, shows that the mounted Drive is gone"""
    if mount_lost(error) or 'Transport endpoint' in str(error or ''):
        raise DriveLost(str(error)) from error
    try:
        os.listdir(archives)
    except OSError as lost:
        if mount_lost(lost):
            raise DriveLost(str(lost)) from error


def commit(api, repo, operations, message, archives, retries=3):
    """
    create_commit, tried again after a failure (the Hub can time out or limit the rate of commits); huggingface_hub
    skips files that are unchanged in the repository, so repeating a commit that went through does no harm
    :return: the commit id
    :raise DriveLost: when the mounted Drive stops answering
    """
    for attempt in range(retries + 1):
        try:
            return api.create_commit(repo, operations=operations, commit_message=message, repo_type=REPO_TYPE).oid
        except Exception as error:                        # noqa: BLE001  (tried again, raised after the last try)
            drive_answers(archives, error)
            if attempt == retries:
                raise
            print(f"commit failed ({type(error).__name__}: {error}); trying again ({attempt + 1} of {retries})",
                  flush=True)
            time.sleep(60 * (attempt + 1))


def commit_front(api, repo, tree, card, index):
    """
    the card (with the licence) and index.csv, committed when the repository does not hold them as they are; in a
    new repository this is the first commit, before any archive
    :param card, index: the contents (bytes)
    :return: the commit id, or None when both are up to date
    """
    from huggingface_hub import CommitOperationAdd
    files = {CARD: card, INDEX: index}
    changed = [name for name, data in files.items() if name not in tree or not same_content(tree[name], data)]
    if not changed:
        return None
    message = "Dataset card, licence and index" if CARD not in tree else f"Update {', '.join(changed)}"
    operations = [CommitOperationAdd(path_in_repo=name, path_or_fileobj=files[name]) for name in changed]
    return api.create_commit(repo, operations=operations, commit_message=message, repo_type=REPO_TYPE).oid


def make_public(api, repo):
    """
    the repository public: HfApi.update_repo_settings(private=False), or update_repo_visibility in versions of
    huggingface_hub that do not have it
    """
    try:
        api.update_repo_settings(repo, private=False, repo_type=REPO_TYPE)
    except (AttributeError, TypeError):
        api.update_repo_visibility(repo, private=False, repo_type=REPO_TYPE)


def prepare(archives, staging, row, tree):
    """
    copies one archive to the staging folder: the MD5 is checked against the index and the SHA-256 kept
    :param tree: the repository listing at the start (repo_tree)
    :return: (row of done.csv, CommitOperationAdd of the staged copy), the operation None when the repository holds
             the archive already
    :raise ValueError: when the MD5 does not match the index (the archive is not uploaded)
    """
    from huggingface_hub import CommitOperationAdd
    staged = staged_path(staging, row)
    md5, sha256, size = stage_archive(Path(archives) / row['run'] / row['file'], staged)
    if md5 != row['md5']:
        raise ValueError(f"MD5 {md5} does not match the index ({row['md5']}); not uploaded")
    done = {'run': row['run'], 'case': row['case'], 'bytes': size, 'md5_ok': True, 'sha256': sha256, 'commit': ''}
    item = tree.get(repo_path(row))
    if item is not None and lfs_of(item) == (size, sha256):
        return done, None
    operation = CommitOperationAdd(path_in_repo=repo_path(row), path_or_fileobj=str(staged))
    info = getattr(operation, 'upload_info', None)      # from huggingface_hub's own read of the staged copy
    if info is not None and (info.sha256.hex() != sha256 or info.size != size):
        raise ValueError("the staged copy differs from the bytes read (SHA-256 of huggingface_hub); not uploaded")
    return done, operation


def compare(tree, rows, done, front):
    """
    the repository against done.csv and the index
    :param tree: the repository listing (repo_tree)
    :param rows: the index rows of the dataset (select_rows)
    :param done: {(run, case): row of done.csv}
    :param front: {path: bytes} of the files next to the archives (the card and index.csv)
    :return: rows of verify.csv; problem is missing, extra, mismatch, or unchecked (an archive in the repository
             whose SHA-256 was never compared with an MD5-checked read, because it is not in done.csv)
    """
    problems = []

    def add(path, problem, detail='', row=None):
        problems.append({'path': path, 'run': row['run'] if row else '', 'case': row['case'] if row else '',
                         'problem': problem, 'detail': detail})

    for name, data in front.items():
        if name not in tree:
            add(name, 'missing')
        elif not same_content(tree[name], data):
            add(name, 'mismatch', "differs from the local file")
    expected = set(front) | {'.gitattributes'}
    for row in rows:
        path = repo_path(row)
        expected.add(path)
        if path not in tree:
            add(path, 'missing', row=row)
            continue
        size, sha256 = lfs_of(tree[path])
        record = done.get((row['run'], row['case']))
        details = []
        if size != int(row['bytes']):
            details.append(f"size {size}, index {row['bytes']}")
        if sha256 is None:
            details.append("no LFS or xet SHA-256 in the listing")
        if record is not None:
            if sha256 is not None and sha256 != record['sha256']:
                details.append(f"SHA-256 {sha256}, done.csv {record['sha256']}")
            if size != int(record['bytes']):
                details.append(f"size {size}, done.csv {record['bytes']}")
        if details:
            add(path, 'mismatch', '; '.join(details), row)
        elif record is None:
            add(path, 'unchecked', "not in done.csv", row)
    for path in sorted(set(tree) - expected):
        add(path, 'extra')
    return problems


def verify(api, repo, rows, done, card, output):
    """compare (one listing of the repository); the problems are printed and written to <output>/verify.csv"""
    tree = repo_tree(api, repo)
    problems = compare(tree, rows, done, {CARD: card, INDEX: index_csv(rows)})
    try:
        with open(Path(output) / 'verify.csv', 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=VERIFY_FIELDS)
            writer.writeheader()
            writer.writerows(problems)
    except OSError as error:
        if mount_lost(error):
            raise DriveLost(str(error)) from error
        raise
    counts = {name: sum(p['problem'] == name for p in problems) for name in ('missing', 'extra', 'mismatch',
                                                                              'unchecked')}
    print(f"{len(tree)} files in {repo}, {len(rows)} archives in the index: {counts['missing']} missing, "
          f"{counts['extra']} extra, {counts['mismatch']} mismatched, {counts['unchecked']} unchecked", flush=True)
    return problems


def run(archives, index, cases, repo, output, card=CARD_FILE, batch=100, batch_bytes=10e9, workers=6, limit=None,
        verify_only=False, token=None, staging=None):
    """
    upload every archive of the dataset that is not yet in done.csv, or with verify_only compare the repository
    with done.csv and the index
    :param cases: file with the case names of the dataset (the release's metadata.parquet or metadata.csv)
    :param repo: repository id, <user or org>/<name>; created private, made public after the card is in it
    :param card: the dataset card (docs/hf/raw_README.md)
    :param limit: upload at most this many archives (a test run)
    :param token: Hugging Face token with write access (None: huggingface_hub's login or HF_TOKEN)
    :param staging: local folder (not on Drive) for the copies of one batch
    :return: the rows of verify.csv with verify_only, else the rows added to done.csv
    :raise DriveLost: when the mounted Drive stops answering (the batch is copied again next time)
    """
    from huggingface_hub import HfApi
    output = Path(output)
    staging = Path(staging or Path(tempfile.gettempdir()) / 'hf-upload-staging')
    try:
        output.mkdir(parents=True, exist_ok=True)
        rows, others = select_rows(index, case_names(cases))
        card = Path(card).read_bytes()
        done = {(r['run'], r['case']): r for r in read_rows(output / 'done.csv')}
        if not verify_only:
            check_staging(archives, staging)
    except OSError as error:
        if mount_lost(error):
            raise DriveLost(str(error)) from error
        raise
    api = HfApi(token=token)
    if verify_only:
        return verify(api, repo, rows, done, card, output)

    api.create_repo(repo, repo_type=REPO_TYPE, private=True, exist_ok=True)     # an existing one is not changed
    tree = repo_tree(api, repo)
    front = commit_front(api, repo, tree, card, index_csv(rows))
    if front:
        print(f"card and index committed ({front})", flush=True)
    if api.repo_info(repo, repo_type=REPO_TYPE).private:                       # it holds the card now
        make_public(api, repo)
        print(f"{repo} is public now", flush=True)
    todo = [r for r in rows if (r['run'], r['case']) not in done]
    if limit is not None:
        todo = todo[:limit]
    clear_staging(staging, rows)                          # copies left by a session that stopped
    print(f"{len(rows)} archives of the dataset ({len(others)} live archives of other cases left out), {len(done)} "
          f"done, {len(tree)} files in the repository, {len(todo)} to upload", flush=True)

    def one(row):
        try:
            return (row, *prepare(archives, staging, row, tree), None)
        except Exception as error:                        # noqa: BLE001  (recorded, the other archives go on)
            return row, None, None, error

    written, total_bytes, started = [], 0, time.time()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for part in shards_of(todo, batch, batch_bytes):
            found, uploads, operations, error_rows, lost = [], [], [], [], None
            try:
                for row, done_row, operation, error in pool.map(one, part):
                    if error is not None:
                        if mount_lost(error):
                            lost = error                  # not an error of the archive: it is copied again later
                        else:
                            error_rows.append({'run': row['run'], 'case': row['case'],
                                               'error': f"{type(error).__name__}: {error}"})
                    elif operation is None:
                        found.append(done_row)
                    else:
                        uploads.append(done_row)
                        operations.append(operation)
                if lost is not None:
                    raise DriveLost(str(lost)) from lost
                if operations:
                    message = (f"Add {part[0]['run']}: {uploads[0]['case']} to {uploads[-1]['case']} "
                               f"({len(operations)} archive{'s' if len(operations) > 1 else ''})")
                    oid = commit(api, repo, operations, message, archives)
                    for r in uploads:
                        r['commit'] = oid
            finally:
                clear_staging(staging, part)              # at most one batch on local disk
            try:
                if found or uploads:
                    append_rows(output / 'done.csv', DONE_FIELDS, found + uploads)
                if error_rows:
                    append_rows(output / 'errors.csv', ERROR_FIELDS, error_rows)
            except OSError as error:
                if mount_lost(error):
                    raise DriveLost(str(error)) from error
                raise
            written += found + uploads
            total_bytes += sum(r['bytes'] for r in uploads)
            elapsed = time.time() - started
            print(f"{part[0]['run']:>8s} {part[0]['case']} ... {len(uploads)} uploaded, {len(found)} found in the "
                  f"repository, {len(error_rows)} errors | {total_bytes / 1e9:.1f} GB in {elapsed / 3600:.2f} h "
                  f"({total_bytes / 1e6 / max(elapsed, 1):.0f} MB/s)", flush=True)
    return written


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--archives', help="folder with <run>/<case>.tar.zst (not needed with --verify-only)")
    parser.add_argument('--index', required=True, help="archive_index.csv of the production")
    parser.add_argument('--cases', required=True, help="metadata.parquet or metadata.csv of the release")
    parser.add_argument('--repo', required=True, help="Hugging Face dataset repository, <user or org>/<name>")
    parser.add_argument('--output', required=True)
    parser.add_argument('--card', default=str(CARD_FILE))
    parser.add_argument('--batch', type=int, default=100)
    parser.add_argument('--batch-bytes', type=float, default=10e9)
    parser.add_argument('--workers', type=int, default=6)
    parser.add_argument('--limit', type=int, default=None)
    parser.add_argument('--staging', default=None, help="local folder for the copies of one batch")
    parser.add_argument('--verify-only', action='store_true')
    args = parser.parse_args()
    if not args.verify_only and not args.archives:
        parser.error("--archives is needed for the upload")
    result = run(args.archives, args.index, args.cases, args.repo, args.output, args.card, args.batch,
                 args.batch_bytes, args.workers, args.limit, args.verify_only, staging=args.staging)
    return 1 if args.verify_only and result else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile tools/upload_release.py
"""
Upload of the release (arrays and tables) to a Hugging Face dataset repository, read where it is on Google Drive:

    python -m tools.upload_release --release <release folder> --repo <user or org>/ms-cofd-2dsl --output <folder> \
        [--md5 <MD5 of checksums.md5>] [--batch N] [--limit N]
    python -m tools.upload_release --release ... --repo ... --output ... --verify-only

Meant for Google Colab with Drive mounted (tools/colab/upload_release.ipynb), like tools/upload_raw_archives.py, whose
staging, commit and listing functions it uses. Every file of the release's checksums.md5 is uploaded at its path in the
release, so `md5sum -c checksums.md5` works in a download of the repository.

The repository:
    README.md               the dataset card (docs/hf/release_README.md), with the licence
    checksums.md5           the release's checksums.md5, unchanged
    DATACARD.md, ...        every file listed in checksums.md5, at its path (<run>/<split>.h5, metadata.parquet, ...)

The MD5 of checksums.md5 is checked first (--md5), so only the release it identifies is uploaded. A new repository is
created private, and the card and checksums.md5 are its first commit. The run never changes its visibility: it is made
public by hand, on its settings page, once the comparison at the end of a run (the listing of the repository against
checksums.md5, done.csv and the card) reports no problem. A card that differs from the one in the repository is
committed again at the next start.

The files follow in commits (HfApi.create_commit) of at most batch files and batch_bytes of one folder; a larger file
goes alone. The top-level files go first, then the runs from the smallest grid to the largest. Each file is read from
Drive once: it is copied to a local staging folder, and its MD5 (checked against checksums.md5) and SHA-256 are
computed in the same pass. The commit uploads the staged copy, so the uploaded bytes are the checked ones;
huggingface_hub's own SHA-256 of the copy is compared with ours. A file whose MD5 does not match is not uploaded and
goes to errors.csv. The staged copies of a batch are removed when its commit is done (or has failed), so at most one
batch is on local disk; a batch that does not fit into the free space of the staging folder stops the run before it
is copied. Nothing in the release folder is changed.

Output (a new folder):
    <output>/done.csv       path, bytes, md5_ok, sha256, commit: one row per file in the repository (commit is empty
                            for a file that was found in the repository)
    <output>/errors.csv     path, error
    <output>/verify.csv     path, problem, detail (--verify-only, and at the end of every run)

A file in done.csv is skipped when the script runs again, so a disconnected Colab session resumes where it stopped.
A file the repository holds that done.csv does not (a commit that went through while the session stopped) is read
and recorded without uploading it again when its size and SHA-256 match. When the Drive mount drops, the run stops
with DriveLost: a partly copied file is removed, and the batch is copied again at the next start.

--verify-only lists the repository with the LFS metadata of its files (size and SHA-256, also for files stored with
xet) and compares every file with done.csv and checksums.md5. Missing, extra and mismatched files, and files in the
repository but not in done.csv (unchecked), are printed and written to verify.csv. Nothing is uploaded.

Needs huggingface_hub >= 0.30 (hf_xet is used when it is installed). The token is that of run(token=...), or of
huggingface_hub's login or HF_TOKEN; it is never written or printed.
"""

import argparse
import csv
import hashlib
import shutil
import sys
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

from tools.extract_line_samples import DriveLost, append_rows, mount_lost, read_rows, shards_of
from tools.upload_raw_archives import (REPO_TYPE, check_staging, commit, drive_answers, lfs_of, repo_tree, same_content,
                                       stage_archive)

DONE_FIELDS = ['path', 'bytes', 'md5_ok', 'sha256', 'commit']
ERROR_FIELDS = ['path', 'error']
VERIFY_FIELDS = ['path', 'problem', 'detail']
CARD, CHECKSUMS = 'README.md', 'checksums.md5'
RUN_ORDER = ['0.25x', 'base_fo', 'base_lu', '4x', '16x']      # smallest grid first
CARD_FILE = Path(__file__).resolve().parent.parent / 'docs' / 'hf' / 'release_README.md'
SPACE_MARGIN = 2e9                                            # bytes left free on the staging disk


def read_checksums(release, md5=None):
    """
    the release's checksums.md5
    :param md5: the expected MD5 of checksums.md5 (None: not checked)
    :return: (its bytes, rows sorted for the upload: path, run (its folder, '' at the top level), md5, bytes)
    :raise ValueError: when its MD5 differs, a path is not a plain relative path, or a listed file is missing
    """
    release = Path(release)
    data = (release / CHECKSUMS).read_bytes()
    if md5 is not None and hashlib.md5(data).hexdigest() != md5:
        raise ValueError(f"{CHECKSUMS} has the MD5 {hashlib.md5(data).hexdigest()}, not {md5}: not this release")
    rows = []
    for line in data.decode().splitlines():
        if not line.strip():
            continue
        checksum, path = line.split(None, 1)
        path = path.lstrip('*')                               # md5sum's binary mark
        parts = path.split('/')
        if path.startswith('/') or '..' in parts or path in (CARD, CHECKSUMS) or len(parts) > 2:
            raise ValueError(f"unexpected path in {CHECKSUMS}: {path}")
        file = release / path
        if not file.is_file():
            raise ValueError(f"{path} is listed in {CHECKSUMS} but not in {release}")
        rows.append({'path': path, 'run': parts[0] if len(parts) == 2 else '', 'md5': checksum,
                     'bytes': file.stat().st_size})

    def order(row):
        run = row['run']
        return (run != '', RUN_ORDER.index(run) if run in RUN_ORDER else len(RUN_ORDER), run, row['path'])
    return data, sorted(rows, key=order)


def staged_path(staging, row):
    return Path(staging) / row['path']


def clear_staging(staging, rows):
    """removes the staged and partial copies of these rows, and the folders they leave empty"""
    for row in rows:
        path = staged_path(staging, row)
        path.unlink(missing_ok=True)
        path.with_name(path.name + '.part').unlink(missing_ok=True)
    for run in {row['run'] for row in rows if row['run']}:
        try:
            (Path(staging) / run).rmdir()
        except OSError:                                       # not empty, or not there
            pass


def check_space(staging, rows):
    """the batch must fit into the free space of the staging disk, with SPACE_MARGIN to spare"""
    Path(staging).mkdir(parents=True, exist_ok=True)
    free, need = shutil.disk_usage(staging).free, sum(int(r['bytes']) for r in rows)
    if need + SPACE_MARGIN > free:
        raise ValueError(f"the batch of {rows[0]['path']} needs {need / 1e9:.1f} GB, the staging folder has "
                         f"{free / 1e9:.1f} GB free ({SPACE_MARGIN / 1e9:.0f} GB are kept free)")


def commit_front(api, repo, tree, card, checksums):
    """
    the card (with the licence) and checksums.md5, committed when the repository does not hold them as they are; in a
    new repository this is the first commit
    :param card, checksums: the contents (bytes)
    :return: the commit id, or None when both are up to date
    """
    from huggingface_hub import CommitOperationAdd
    files = {CARD: card, CHECKSUMS: checksums}
    changed = [name for name, data in files.items() if name not in tree or not same_content(tree[name], data)]
    if not changed:
        return None
    message = "Dataset card, licence and checksums" if CARD not in tree else f"Update {', '.join(changed)}"
    operations = [CommitOperationAdd(path_in_repo=name, path_or_fileobj=files[name]) for name in changed]
    return api.create_commit(repo, operations=operations, commit_message=message, repo_type=REPO_TYPE).oid


def prepare(release, staging, row, tree):
    """
    copies one file to the staging folder: the MD5 is checked against checksums.md5 and the SHA-256 kept
    :param tree: the repository listing at the start (repo_tree)
    :return: (row of done.csv, CommitOperationAdd of the staged copy), the operation None when the repository holds
             the file already
    :raise ValueError: when the MD5 does not match checksums.md5 (the file is not uploaded)
    """
    from huggingface_hub import CommitOperationAdd
    staged = staged_path(staging, row)
    md5, sha256, size = stage_archive(Path(release) / row['path'], staged)
    if md5 != row['md5']:
        raise ValueError(f"MD5 {md5} does not match {CHECKSUMS} ({row['md5']}); not uploaded")
    done = {'path': row['path'], 'bytes': size, 'md5_ok': True, 'sha256': sha256, 'commit': ''}
    item = tree.get(row['path'])
    if item is not None and (lfs_of(item) == (size, sha256) or lfs_of(item) == (size, None)
                             and same_content(item, staged.read_bytes())):   # a small file stored without LFS
        return done, None
    operation = CommitOperationAdd(path_in_repo=row['path'], path_or_fileobj=str(staged))
    info = getattr(operation, 'upload_info', None)          # from huggingface_hub's own read of the staged copy
    if info is not None and (info.sha256.hex() != sha256 or info.size != size):
        raise ValueError("the staged copy differs from the bytes read (SHA-256 of huggingface_hub); not uploaded")
    return done, operation


def compare(tree, rows, done, front, content=None):
    """
    the repository against done.csv and checksums.md5
    :param tree: the repository listing (repo_tree)
    :param rows: the files of checksums.md5 (read_checksums)
    :param done: {path: row of done.csv}
    :param front: {path: bytes} of the card and checksums.md5
    :param content: function of a path that returns the release file's bytes, for the small files the Hub stores
                    without LFS (their listing has a git blob id, no SHA-256); None: such a file is a mismatch
    :return: rows of verify.csv; problem is missing, extra, mismatch, or unchecked (a file in the repository whose
             SHA-256 was never compared with an MD5-checked read, because it is not in done.csv)
    """
    problems = []

    def add(path, problem, detail=''):
        problems.append({'path': path, 'problem': problem, 'detail': detail})

    for name, data in front.items():
        if name not in tree:
            add(name, 'missing')
        elif not same_content(tree[name], data):
            add(name, 'mismatch', "differs from the local file")
    expected = set(front) | {'.gitattributes'}
    for row in rows:
        path = row['path']
        expected.add(path)
        if path not in tree:
            add(path, 'missing')
            continue
        size, sha256 = lfs_of(tree[path])
        record = done.get(path)
        details = []
        if size != int(row['bytes']):
            details.append(f"size {size}, release {row['bytes']}")
        elif sha256 is None:
            if content is None:
                details.append("no LFS or xet SHA-256 in the listing")
            elif not same_content(tree[path], content(path)):
                details.append("git blob differs from the release file")
        if record is not None:
            if sha256 is not None and sha256 != record['sha256']:
                details.append(f"SHA-256 {sha256}, done.csv {record['sha256']}")
            if size != int(record['bytes']):
                details.append(f"size {size}, done.csv {record['bytes']}")
        if details:
            add(path, 'mismatch', '; '.join(details))
        elif record is None:
            add(path, 'unchecked', "not in done.csv")
    for path in sorted(set(tree) - expected):
        add(path, 'extra')
    return problems


def verify(api, repo, rows, done, front, output, release):
    """
    compare (one listing of the repository); the problems are printed and written to <output>/verify.csv
    :param release: the release folder, read for the small files the Hub stores without LFS
    """
    tree = repo_tree(api, repo)
    problems = compare(tree, rows, done, front, content=lambda path: (Path(release) / path).read_bytes())
    try:
        with open(Path(output) / 'verify.csv', 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=VERIFY_FIELDS)
            writer.writeheader()
            writer.writerows(problems)
    except OSError as error:
        if mount_lost(error):
            raise DriveLost(str(error)) from error
        raise
    counts = {name: sum(p['problem'] == name for p in problems) for name in ('missing', 'extra', 'mismatch',
                                                                              'unchecked')}
    print(f"{len(tree)} files in {repo}, {len(rows)} files in {CHECKSUMS}: {counts['missing']} missing, "
          f"{counts['extra']} extra, {counts['mismatch']} mismatched, {counts['unchecked']} unchecked", flush=True)
    return problems


def run(release, repo, output, card=CARD_FILE, md5=None, batch=20, batch_bytes=10e9, workers=4, limit=None,
        verify_only=False, token=None, staging=None):
    """
    upload every file of the release that is not yet in done.csv, then compare the repository with the release; with
    verify_only only the comparison
    :param release: the release folder (holds checksums.md5)
    :param repo: repository id, <user or org>/<name>; created private (made public by hand)
    :param card: the dataset card (docs/hf/release_README.md)
    :param md5: the expected MD5 of checksums.md5 (None: not checked)
    :param limit: upload at most this many files (a test run)
    :param token: Hugging Face token with write access (None: huggingface_hub's login or HF_TOKEN)
    :param staging: local folder (not on Drive) for the copies of one batch
    :return: the rows of verify.csv
    :raise DriveLost: when the mounted Drive stops answering (the batch is copied again next time)
    """
    from huggingface_hub import HfApi
    output = Path(output)
    staging = Path(staging or Path(tempfile.gettempdir()) / 'hf-upload-staging')
    try:
        output.mkdir(parents=True, exist_ok=True)
        checksums, rows = read_checksums(release, md5)
        card = Path(card).read_bytes()
        done = {r['path']: r for r in read_rows(output / 'done.csv')}
        if not verify_only:
            check_staging(release, staging)
    except OSError as error:
        if mount_lost(error):
            raise DriveLost(str(error)) from error
        raise
    front = {CARD: card, CHECKSUMS: checksums}
    api = HfApi(token=token)
    if verify_only:
        return verify(api, repo, rows, done, front, output, release)

    api.create_repo(repo, repo_type=REPO_TYPE, private=True, exist_ok=True)     # an existing one is not changed
    tree = repo_tree(api, repo)
    first = commit_front(api, repo, tree, card, checksums)
    if first:
        print(f"card and {CHECKSUMS} committed ({first})", flush=True)
    todo = [r for r in rows if r['path'] not in done]
    if limit is not None:
        todo = todo[:limit]
    clear_staging(staging, rows)                              # copies left by a session that stopped
    print(f"{len(rows)} files in {CHECKSUMS} ({sum(r['bytes'] for r in rows) / 1e9:.1f} GB), {len(done)} done, "
          f"{len(tree)} files in the repository, {len(todo)} to upload", flush=True)

    def one(row):
        try:
            return (row, *prepare(release, staging, row, tree), None)
        except Exception as error:                            # noqa: BLE001  (recorded, the other files go on)
            return row, None, None, error

    total_bytes, started, failed = 0, time.time(), False
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for part in shards_of(todo, batch, batch_bytes):
            check_space(staging, part)
            found, uploads, operations, error_rows, lost = [], [], [], [], None
            try:
                for row, done_row, operation, error in pool.map(one, part):
                    if error is not None:
                        if mount_lost(error):
                            lost = error                      # not an error of the file: it is copied again later
                        else:
                            error_rows.append({'path': row['path'], 'error': f"{type(error).__name__}: {error}"})
                    elif operation is None:
                        found.append(done_row)
                    else:
                        uploads.append(done_row)
                        operations.append(operation)
                if lost is not None:
                    raise DriveLost(str(lost)) from lost
                if operations:
                    names = [r['path'] for r in uploads]
                    message = (f"Add {names[0]}" if len(names) == 1 else f"Add {len(names)} files of {part[0]['run']}/"
                               if part[0]['run'] else f"Add {len(names)} top-level files")
                    oid = commit(api, repo, operations, message, release)
                    for r in uploads:
                        r['commit'] = oid
            finally:
                clear_staging(staging, part)                  # at most one batch on local disk
            try:
                if found or uploads:
                    append_rows(output / 'done.csv', DONE_FIELDS, found + uploads)
                if error_rows:
                    append_rows(output / 'errors.csv', ERROR_FIELDS, error_rows)
            except OSError as error:
                if mount_lost(error):
                    raise DriveLost(str(error)) from error
                raise
            failed = failed or bool(error_rows)
            for r in found + uploads:
                done[r['path']] = {k: str(v) for k, v in r.items()}
            total_bytes += sum(r['bytes'] for r in uploads)
            elapsed = time.time() - started
            print(f"{part[0]['path']} ... {len(uploads)} uploaded, {len(found)} found in the repository, "
                  f"{len(error_rows)} errors | {total_bytes / 1e9:.1f} GB in {elapsed / 3600:.2f} h "
                  f"({total_bytes / 1e6 / max(elapsed, 1):.0f} MB/s)", flush=True)

    drive_answers(release)
    problems = verify(api, repo, rows, done, front, output, release)
    if problems or failed:
        print(f"{repo} does not match the release yet", flush=True)
    elif api.repo_info(repo, repo_type=REPO_TYPE).private:
        print(f"{repo} holds the whole release and stays private: make it public on its settings page", flush=True)
    else:
        print(f"{repo} holds the whole release", flush=True)
    return problems


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('--release', required=True, help="the release folder, with checksums.md5")
    parser.add_argument('--repo', required=True, help="Hugging Face dataset repository, <user or org>/<name>")
    parser.add_argument('--output', required=True)
    parser.add_argument('--card', default=str(CARD_FILE))
    parser.add_argument('--md5', default=None, help="the expected MD5 of checksums.md5")
    parser.add_argument('--batch', type=int, default=20)
    parser.add_argument('--batch-bytes', type=float, default=10e9)
    parser.add_argument('--workers', type=int, default=4)
    parser.add_argument('--limit', type=int, default=None)
    parser.add_argument('--staging', default=None, help="local folder for the copies of one batch")
    parser.add_argument('--verify-only', action='store_true')
    args = parser.parse_args()
    problems = run(args.release, args.repo, args.output, args.card, args.md5, args.batch, args.batch_bytes,
                   args.workers, args.limit, args.verify_only, staging=args.staging)
    return 1 if problems else 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
import os, time, importlib
assert os.path.isfile(RELEASE + '/checksums.md5'), 'RELEASE is not the release folder'
assert os.path.isfile(CARD), 'put README.md into OUTPUT first'
import tools.extract_line_samples, tools.upload_raw_archives, tools.upload_release
importlib.reload(tools.extract_line_samples)
importlib.reload(tools.upload_raw_archives)
upload = importlib.reload(tools.upload_release)
# the Drive mount can drop in a long run: Drive is mounted again and the run goes on (up to RETRIES times); the batch
# that was being copied is copied again
RETRIES = 20
for attempt in range(RETRIES + 1):
    try:
        upload.run(RELEASE, REPO_ID, OUTPUT, card=CARD, md5=RELEASE_MD5, batch=BATCH, workers=WORKERS, limit=LIMIT,
                   token=hf_token, staging=STAGING)
        break
    except upload.DriveLost as error:
        print(f'Drive mount lost ({error}); mounting again ({attempt + 1} of {RETRIES})', flush=True)
        time.sleep(30)
        drive.mount('/content/drive', force_remount=True)

In [ ]:
# one listing of the repository, compared with done.csv and checksums.md5 (also written to OUTPUT/verify.csv)
problems = upload.run(RELEASE, REPO_ID, OUTPUT, card=CARD, md5=RELEASE_MD5, verify_only=True, token=hf_token)
for p in sorted(problems, key=lambda p: p['problem'] == 'missing')[:20]:
    print(p)

In [ ]:
import csv
done = list(csv.DictReader(open(OUTPUT + '/done.csv'))) if os.path.exists(OUTPUT + '/done.csv') else []
uploaded = {r['path'] for r in done}
errors = list(csv.DictReader(open(OUTPUT + '/errors.csv'))) if os.path.exists(OUTPUT + '/errors.csv') else []
errors = {r['path']: r for r in errors if r['path'] not in uploaded}                      # latest, not uploaded
_, rows = upload.read_checksums(RELEASE, RELEASE_MD5)
print(len(done), 'files in the repository of', len(rows), f'| {sum(int(r["bytes"]) for r in done) / 1e9:.1f} GB',
      '|', sum('MD5' in r['error'] for r in errors.values()), 'MD5 mismatches |', len(errors), 'files with errors')
for r in list(errors.values())[:20]:
    print(r)